# Publication-Ready Challenge Improvements

This notebook is the fresh improvement path for the English-Hindi audio deepfake paper.

It uses the harder `RawCh_*` held-out-generator protocol with `WavLM_embeddings_unified`, then adds:
- exact dataset and feature sanity checks,
- optional multi-seed training,
- prediction for validation/test,
- score-distribution-aware thresholding,
- global, per-language, and per-source evaluation,
- saved ensemble predictions and summary.

Training is disabled by default. Set the flags in the config cell when you are ready to run it.

In [ ]:
from pathlib import Path
from argparse import Namespace
from collections import Counter, defaultdict
import csv
import json
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

ROOT = Path.cwd().resolve()
if not (ROOT / 'safari_wavlm_ensemble_train.py').exists() and not (ROOT / 'src' / 'safari_wavlm_ensemble_train.py').exists():
    ROOT = ROOT.parent
CODE_ROOT = ROOT / 'src' if (ROOT / 'src' / 'safari_wavlm_ensemble_train.py').exists() else ROOT
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from safari_wavlm_ensemble_train import train as train_safari_wavlm
from predict_safari_wavlm_ensemble import device_from_arg, load_ckpt, build_model, run_batch

print('Project root:', ROOT)
print('Code root:', CODE_ROOT)
print('CUDA available:', torch.cuda.is_available())

## 1. Configuration

The default config evaluates existing completed seed outputs. Turn on training/prediction only when you are ready.

In [ ]:
DATA_ROOT = ROOT / 'datasets' if (ROOT / 'datasets').exists() else ROOT

TRAIN_CSV = DATA_ROOT / 'RawCh_train' / 'RawCh_train' / 'balanced_index.csv'
VAL_CSV = DATA_ROOT / 'RawCh_val' / 'RawCh_val' / 'balanced_index.csv'
TEST_CSV = DATA_ROOT / 'RawCh_test' / 'RawCh_test' / 'balanced_index.csv'
FEATURE_ROOT = DATA_ROOT / 'WavLM_embeddings_unified'

ARTIFACT_ROOT = ROOT / 'artifacts' / 'publication_challenge_improved'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

SEEDS = [17, 42, 77, 123, 202]
CURRENT_SEED = 17

RUN_TRAINING = False
RUN_ALL_SEEDS = False
RUN_PREDICTION = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = torch.cuda.is_available()
EPOCHS = 60
BATCH_SIZE = 512
NUM_WORKERS = 4

print('Artifact root:', ARTIFACT_ROOT)
print('Device:', DEVICE, 'AMP:', AMP)

## 2. Dataset and Feature Sanity Checks

In [ ]:
def read_df(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def source_from_audio_path(value: str) -> str:
    parts = Path(str(value)).parts
    if 'Raw Dataset' in parts:
        i = parts.index('Raw Dataset')
        return '/'.join(parts[i + 1:i + 3])
    return '<unknown>'

def summarize_split(name: str, csv_path: Path, feature_root: Path) -> pd.DataFrame:
    df = read_df(csv_path)
    print(f'[{name}] rows={len(df)} columns={list(df.columns)}')
    print('label counts:', df['label'].value_counts().to_dict())
    print('language counts:', df['language'].value_counts().to_dict())
    print('language x label:')
    print(pd.crosstab(df['language'], df['label']))
    missing = [p for p in df['feature_path'] if not (feature_root / p).exists()]
    print('missing feature files:', len(missing))
    if 'audio_path' in df.columns:
        print('source counts:')
        print(df['audio_path'].map(source_from_audio_path).value_counts())
    print()
    return df

train_df = summarize_split('train', TRAIN_CSV, FEATURE_ROOT)
val_df = summarize_split('val', VAL_CSV, FEATURE_ROOT)
test_df = summarize_split('test', TEST_CSV, FEATURE_ROOT)

sample_feature = FEATURE_ROOT / train_df.iloc[0]['feature_path']
x = np.load(sample_feature)
print('sample feature:', sample_feature)
print('sample shape:', x.shape, 'mean:', float(x.mean()), 'std:', float(x.std()))

## 3. Optional Training

This calls the project trainer directly, not shell commands. The trainer now has the improved threshold search.

In [ ]:
def seed_dir(seed: int) -> Path:
    d = ARTIFACT_ROOT / f'seed_{seed}'
    d.mkdir(parents=True, exist_ok=True)
    return d

def make_train_args(seed: int) -> Namespace:
    return Namespace(
        project_root=str(ROOT),
        train_split='RawCh_train',
        val_split='RawCh_val',
        test_split='RawCh_test',
        train_csv=str(TRAIN_CSV),
        val_csv=str(VAL_CSV),
        test_csv=str(TEST_CSV),
        train_feature_dir=str(FEATURE_ROOT),
        val_feature_dir=str(FEATURE_ROOT),
        test_feature_dir=str(FEATURE_ROOT),
        wavlm_train_feature_dir=str(FEATURE_ROOT),
        wavlm_val_feature_dir=str(FEATURE_ROOT),
        wavlm_test_feature_dir=str(FEATURE_ROOT),
        out_dir=str(seed_dir(seed)),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        seed=seed,
        hidden_dim=384,
        language_emb_dim=32,
        lr=3e-4,
        max_lr=2e-3,
        weight_decay=1e-4,
        patience=8,
        view_noise_std=0.01,
        view_drop_prob=0.03,
        stats_samples=50000,
        device=DEVICE,
        amp=AMP,
    )

def train_one_seed(seed: int) -> Path:
    print(f'Training seed {seed}')
    train_safari_wavlm(make_train_args(seed))
    return seed_dir(seed) / 'best_safari_wavlm_ensemble.pt'

if RUN_TRAINING:
    seeds_to_run = SEEDS if RUN_ALL_SEEDS else [CURRENT_SEED]
    for seed in seeds_to_run:
        train_one_seed(seed)
else:
    print('Training skipped. Set RUN_TRAINING = True when ready.')

## 4. Optional Prediction

Run this after training, or leave it off if prediction CSVs already exist.

In [ ]:
def predict_seed_split(seed: int, split: str) -> Path:
    d = seed_dir(seed)
    model_path = d / 'best_safari_wavlm_ensemble.pt'
    if not model_path.exists():
        raise FileNotFoundError(f'Missing checkpoint for seed {seed}: {model_path}')

    input_csv = {'val': VAL_CSV, 'test': TEST_CSV}[split]
    output_csv = d / f'{split}_predictions.csv'
    device_obj = device_from_arg(DEVICE)
    ckpt = load_ckpt(model_path, device_obj)
    model = build_model(ckpt, device_obj)
    run_batch(
        model=model,
        ckpt=ckpt,
        input_csv=input_csv,
        base_feature_root=FEATURE_ROOT,
        wavlm_feature_root=FEATURE_ROOT,
        output_csv=output_csv,
        device=device_obj,
    )
    return output_csv

if RUN_PREDICTION:
    seeds_to_predict = SEEDS if RUN_ALL_SEEDS else [CURRENT_SEED]
    for seed in seeds_to_predict:
        predict_seed_split(seed, 'val')
        predict_seed_split(seed, 'test')
else:
    print('Prediction skipped. Set RUN_PREDICTION = True when ready.')

## 5. Collect Completed Seeds and Ensemble

In [ ]:
def completed_seeds() -> list[int]:
    done = []
    for seed in SEEDS:
        d = seed_dir(seed)
        if (d / 'val_predictions.csv').exists() and (d / 'test_predictions.csv').exists():
            done.append(seed)
    return done

def read_predictions(path: Path) -> dict[str, dict]:
    with path.open('r', encoding='utf-8', newline='') as handle:
        return {row['feature_path']: row for row in csv.DictReader(handle)}

def aggregate_seed_predictions(seeds: list[int], split: str) -> pd.DataFrame:
    tables = []
    for seed in seeds:
        path = seed_dir(seed) / f'{split}_predictions.csv'
        tables.append(read_predictions(path))
    common_keys = sorted(set.intersection(*(set(t.keys()) for t in tables)))
    rows = []
    for key in common_keys:
        base = dict(tables[0][key])
        probs = [float(t[key]['fake_probability']) for t in tables]
        base['fake_probability'] = float(np.mean(probs))
        rows.append(base)
    return pd.DataFrame(rows)

done = completed_seeds()
print('Completed seeds in this notebook artifact root:', done)

# Fallback: evaluate the existing seed_17 challenge run if no new seeds are present.
if not done:
    existing = ROOT / 'artifacts' / 'multiseed_stepwise_challenge' / 'seed_17'
    if (existing / 'val_predictions.csv').exists() and (existing / 'test_predictions.csv').exists():
        print('No new completed seeds found. Loading existing challenge seed_17 predictions for analysis.')
        val_ens = pd.read_csv(existing / 'val_predictions.csv')
        test_ens = pd.read_csv(existing / 'test_predictions.csv')
        done = ['existing_seed_17']
    else:
        raise RuntimeError('No completed predictions found. Train/predict at least one seed first.')
else:
    val_ens = aggregate_seed_predictions(done, 'val')
    test_ens = aggregate_seed_predictions(done, 'test')

print('val rows:', len(val_ens), 'test rows:', len(test_ens))
val_ens.head()

## 6. Improved Thresholding and Detailed Evaluation

In [ ]:
def to_binary_label(series: pd.Series) -> np.ndarray:
    return series.astype(str).str.lower().eq('fake').astype(int).to_numpy()

def threshold_candidates(scores: np.ndarray) -> np.ndarray:
    return np.unique(
        np.concatenate([
            np.linspace(0.0, 1.0, 1001),
            np.quantile(scores, np.linspace(0.0, 1.0, 1001)),
        ])
    )

def find_best_threshold(y_true: np.ndarray, scores: np.ndarray) -> tuple[float, dict]:
    best = (-1.0, -1.0, 0.5)
    best_metrics = {}
    for t in threshold_candidates(scores):
        pred = (scores >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        acc = accuracy_score(y_true, pred)
        if (f1, acc) > (best[0], best[1]):
            best = (float(f1), float(acc), float(t))
            best_metrics = compute_metrics(y_true, scores, float(t))
    return best[2], best_metrics

def compute_metrics(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    pred = (scores >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred, average='binary', zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        'n': int(len(y_true)),
        'threshold': float(threshold),
        'roc_auc': float(roc_auc_score(y_true, scores)) if len(set(y_true)) > 1 else float('nan'),
        'accuracy': float(accuracy_score(y_true, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, pred)),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

def evaluate_table(df: pd.DataFrame, threshold: float) -> dict:
    y = to_binary_label(df['label'])
    scores = df['fake_probability'].astype(float).to_numpy()
    return compute_metrics(y, scores, threshold)

def grouped_metrics(df: pd.DataFrame, threshold: float, group_col: str) -> pd.DataFrame:
    rows = []
    for name, sub in df.groupby(group_col):
        y = to_binary_label(sub['label'])
        scores = sub['fake_probability'].astype(float).to_numpy()
        m = compute_metrics(y, scores, threshold) if len(set(y)) > 1 else evaluate_single_class(sub, threshold)
        m[group_col] = name
        rows.append(m)
    return pd.DataFrame(rows).sort_values('n', ascending=False)

def evaluate_single_class(df: pd.DataFrame, threshold: float) -> dict:
    y = to_binary_label(df['label'])
    scores = df['fake_probability'].astype(float).to_numpy()
    pred = (scores >= threshold).astype(int)
    return {
        'n': int(len(df)),
        'threshold': float(threshold),
        'roc_auc': float('nan'),
        'accuracy': float(accuracy_score(y, pred)),
        'balanced_accuracy': float('nan'),
        'precision': float('nan'),
        'recall': float('nan'),
        'f1': float('nan'),
        'tn': int(((y == 0) & (pred == 0)).sum()),
        'fp': int(((y == 0) & (pred == 1)).sum()),
        'fn': int(((y == 1) & (pred == 0)).sum()),
        'tp': int(((y == 1) & (pred == 1)).sum()),
        'mean_fake_probability': float(scores.mean()),
    }

val_y = to_binary_label(val_ens['label'])
val_scores = val_ens['fake_probability'].astype(float).to_numpy()
best_t, val_best_metrics = find_best_threshold(val_y, val_scores)

print('Best validation threshold:', best_t)
print('Validation metrics:', val_best_metrics)
print('Test metrics:', evaluate_table(test_ens, best_t))

In [ ]:
print('Per-language test metrics')
display(grouped_metrics(test_ens, best_t, 'language'))

if 'audio_path' in test_ens.columns:
    test_with_source = test_ens.copy()
    test_with_source['source'] = test_with_source['audio_path'].map(source_from_audio_path)
    print('Per-source test metrics')
    display(grouped_metrics(test_with_source, best_t, 'source'))

## 7. Save Improved Outputs

In [ ]:
def add_predictions(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    out = df.copy()
    out['fake_probability'] = out['fake_probability'].astype(float)
    out['predicted_label'] = np.where(out['fake_probability'] >= threshold, 'fake', 'real')
    return out

out_dir = ARTIFACT_ROOT / 'ensemble_predictions'
out_dir.mkdir(parents=True, exist_ok=True)
val_out = out_dir / 'val_improved_predictions.csv'
test_out = out_dir / 'test_improved_predictions.csv'

val_saved = add_predictions(val_ens, best_t)
test_saved = add_predictions(test_ens, best_t)
val_saved.to_csv(val_out, index=False)
test_saved.to_csv(test_out, index=False)

summary = {
    'protocol': 'RawCh held-out-generator challenge',
    'model': 'SafariWavLMEnsemble',
    'feature_root': str(FEATURE_ROOT),
    'seeds': done,
    'threshold': float(best_t),
    'val_metrics': evaluate_table(val_saved, best_t),
    'test_metrics': evaluate_table(test_saved, best_t),
    'val_predictions': str(val_out),
    'test_predictions': str(test_out),
}

summary_path = ARTIFACT_ROOT / 'publication_challenge_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Saved:', summary_path)
print(json.dumps(summary, indent=2))

## 8. What To Look For

For publication readiness, the target is not only high AUC. Check these before claiming the model is final:

- Test AUC >= 0.95 on `RawCh_test`.
- Test accuracy >= 0.90 on `RawCh_test`.
- English OpenAI and xTTS fake recall improves versus the old seed-17 run.
- Hindi remains strong without relying on a single easy source.
- Full 5-seed ensemble is reported, not only one seed.
- Include ablations against balanced embeddings, strict split, and challenge split.